# Tugas 3 Visi Komputer — Segmentasi Instance Ruangan Indoor
**Muhammad Fathur Aziz — 5054241043** · Evaluasi CPMK 3

| Komponen | Isi |
|---|---|
| Data latih | *Panoptic Indoor Segmentation* v5 (Roboflow Universe) |
| Model (3 model, 2 keluarga) | **YOLO11n-seg** & **YOLOv8n-seg** (one-stage CNN, mask = kombinasi *prototype* 160×160) vs **Mask R-CNN R50-FPN v2** (two-stage CNN, mask per-RoI 28×28) |
| Variabel NRP (digit terakhir **3**) | **Jumlah data latih: 25% / 50% / 100%** |
| Test set | (1) test set asli dataset, (2) ≥40 foto ruangan buatan sendiri (≥10 kondisi sulit) |
| Metrik | Mask mAP@50:95 dan mAP@50 (COCOeval `segm`, **pipeline evaluasi yang sama untuk ketiga model**) |

Jalankan di Google Colab dengan GPU (*Runtime → Change runtime type → T4 GPU*), urut dari atas. Setiap training dibatasi **maks 2 jam GPU** dan waktunya otomatis dicatat ke `bukti/gpu_time_log.csv`.

## 0. Persiapan

In [ ]:
import os, sys
MODE_UJI = os.environ.get('T3_SMOKE') == '1'   # hanya untuk uji cepat lokal dengan data sintetis
IN_COLAB = 'google.colab' in sys.modules
if not MODE_UJI:
    %pip install -q ultralytics roboflow pycocotools

In [ ]:
import torch   # di Windows, torch harus di-import sebelum cv2/pandas (kalau tidak: WinError 1114 c10.dll)
import json, time, random, glob, shutil, csv, datetime, collections, warnings
import numpy as np, pandas as pd, cv2, yaml
import matplotlib.pyplot as plt
from contextlib import contextmanager, redirect_stdout
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskU
import io

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
print('Perangkat:', GPU_NAME)
if DEVICE == 'cpu' and not MODE_UJI:
    print('Belum pakai GPU! Di Colab: Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.')

### 0.1 Konfigurasi
Isi `RF_DATASET` (slug dataset Panoptic Indoor Segmentation v5 di Roboflow Universe — lihat URL: `universe.roboflow.com/<workspace>/<project>/dataset/5`) dan `RF_OWN` (project Roboflow milikmu yang berisi foto ruangan + anotasi).

API key Roboflow: simpan di **Colab Secrets** (ikon kunci di sidebar kiri) dengan nama `ROBOFLOW_API_KEY`, jangan ditulis di notebook.

In [ ]:
UJI_CEPAT = False   # True = uji alur (2 epoch, folder terpisah). Ganti ke False untuk eksperimen sungguhan!
NRP, SEED = '5054241043', 0
RF_DATASET = dict(workspace='panopticindoor', project='panoptic-indoor-segmentation', version=5)  # 922 citra: train 397 / valid 166 / test 359
RF_OWN     = dict(workspace='5054241043-student-its-ac-id', project='test-ruangan-5054241043', version=1)  # test set buatanmu (cek nomor version setelah Generate)

IMGSZ          = 640
EPOCHS_YOLO    = 60      # dihentikan lebih awal oleh early-stopping (patience) atau batas 2 jam
EPOCHS_MRCNN   = 24
BATCH_YOLO     = 16
BATCH_MRCNN    = 4
BATAS_JAM_GPU  = 2.0     # ketentuan tugas: maks 2 jam GPU Colab per model
FRACTIONS      = [1.0, 0.5, 0.25]   # variabel NRP (digit 3): jumlah data latih -> diterapkan pada KETIGA model
MRCNN_FRACTIONS = FRACTIONS          # Mask R-CNN ikut 100/50/25% (ketentuan: tabel & grafik untuk ketiga model)

# Folder hasil: di Colab disimpan ke Google Drive supaya tidak hilang saat runtime putus.
OUT = 'hasil_tugas3'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/Tugas3_Segmentasi'
if UJI_CEPAT and not MODE_UJI:
    OUT += '_uji'
    EPOCHS_YOLO, EPOCHS_MRCNN = 2, 1
    print('MODE UJI CEPAT: 2 epoch, hasil di', OUT, '-> ganti UJI_CEPAT = False untuk run sungguhan')
if MODE_UJI:
    OUT = os.environ['T3_OUT']
    IMGSZ, EPOCHS_YOLO, EPOCHS_MRCNN, BATCH_YOLO, BATCH_MRCNN = 160, 2, 2, 4, 2
WORK = os.path.abspath('data_tugas3')     # dataset di disk lokal (lebih cepat dari Drive)
RUNS, BUKTI = f'{OUT}/runs', f'{OUT}/bukti'
for d in (WORK, RUNS, BUKTI, f'{OUT}/gambar'):
    os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Hasil disimpan di:', OUT)

### 0.2 Pencatat waktu GPU (bukti proses)

In [ ]:
GPU_LOG = f'{BUKTI}/gpu_time_log.csv'

@contextmanager
def catat_waktu_gpu(model, eksperimen, catatan=''):
    t0, mulai, status = time.time(), datetime.datetime.now(), 'selesai'
    try:
        yield
    except BaseException as e:
        status = f'gagal/terhenti ({type(e).__name__})'
        raise
    finally:
        menit = (time.time() - t0) / 60
        baru = not os.path.exists(GPU_LOG)
        with open(GPU_LOG, 'a', newline='') as f:
            w = csv.writer(f)
            if baru:
                w.writerow(['mulai', 'selesai', 'model', 'eksperimen', 'durasi_menit', 'perangkat', 'status', 'catatan'])
            w.writerow([mulai.isoformat(timespec='seconds'), datetime.datetime.now().isoformat(timespec='seconds'),
                        model, eksperimen, round(menit, 2), GPU_NAME, status, catatan])
        print(f'[GPU log] {model} / {eksperimen}: {menit:.1f} menit ({status})')

def waktu_tercatat():
    return pd.read_csv(GPU_LOG) if os.path.exists(GPU_LOG) else pd.DataFrame()

## 1. Dataset
### 1.1 Unduh dataset latih & test set buatan sendiri
Keduanya diekspor dalam format **YOLOv8 (polygon)**. Test set buatan sendiri digabung dari semua split project-mu, dan ID kelasnya dipetakan ulang **berdasarkan nama** agar sama dengan dataset latih.

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
# path dinormalisasi (abspath) agar pemisah folder konsisten, termasuk di Windows
list_images = lambda d: sorted(os.path.abspath(p) for p in glob.glob(f'{d}/*') if p.lower().endswith(IMG_EXT)) if os.path.isdir(d) else []
def label_of(p):
    p = os.path.abspath(p); a, b = os.path.split(os.path.dirname(p))
    assert b == 'images', f'citra harus berada di folder images/: {p}'
    return os.path.join(a, 'labels', os.path.splitext(os.path.basename(p))[0] + '.txt')
norm = lambda s: str(s).strip().lower()

def ambil_key():
    if MODE_UJI: return None
    try:
        from google.colab import userdata
        return userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        import getpass
        return os.environ.get('ROBOFLOW_API_KEY') or getpass.getpass('Roboflow API key: ')

def unduh_roboflow(cfg, folder):
    if os.path.exists(f'{folder}/data.yaml'):
        return folder
    from roboflow import Roboflow
    rf = Roboflow(api_key=ambil_key())
    rf.workspace(cfg['workspace']).project(cfg['project']).version(cfg['version']).download('yolov8', location=folder)
    return folder

if MODE_UJI:
    DS_DIR, OWN_RAW = os.environ['T3_DS'], os.environ['T3_OWN']
else:
    DS_DIR = unduh_roboflow(RF_DATASET, f'{WORK}/panoptic_indoor_v5')
    OWN_RAW = unduh_roboflow(RF_OWN, f'{WORK}/own_raw')

meta = yaml.safe_load(open(f'{DS_DIR}/data.yaml'))
NAMES = meta['names'] if isinstance(meta['names'], list) else [meta['names'][k] for k in sorted(meta['names'])]
NC = len(NAMES)
SPLIT = {s: f'{DS_DIR}/{d}/images' for s, d in [('train', 'train'), ('val', 'valid'), ('test', 'test')]}
if not list_images(SPLIT['test']):
    warnings.warn('Dataset tidak punya split test -> split valid dipakai sebagai test asli. Catat ini di laporan!')
    SPLIT['test'] = SPLIT['val']
print(f'{NC} kelas:', NAMES)

# --- gabungkan test set buatan sendiri + petakan ulang kelas berdasarkan nama ---
OWN_DIR = f'{WORK}/own_test'
shutil.rmtree(OWN_DIR, ignore_errors=True)
os.makedirs(f'{OWN_DIR}/images'); os.makedirs(f'{OWN_DIR}/labels')
own_meta = yaml.safe_load(open(f'{OWN_RAW}/data.yaml'))
own_names = own_meta['names'] if isinstance(own_meta['names'], list) else [own_meta['names'][k] for k in sorted(own_meta['names'])]
idx = {norm(n): i for i, n in enumerate(NAMES)}
peta = {i: idx.get(norm(n)) for i, n in enumerate(own_names)}
tak_dikenal = [own_names[i] for i, j in peta.items() if j is None]
if tak_dikenal:
    print('PERINGATAN: kelas di test set-mu yang tidak ada di dataset latih (anotasinya diabaikan):', tak_dikenal)
for sub in ('train', 'valid', 'test'):
    for p in list_images(f'{OWN_RAW}/{sub}/images'):
        shutil.copy(p, f'{OWN_DIR}/images/')
        baris = []
        if os.path.exists(label_of(p)):
            for l in open(label_of(p)).read().split('\n'):
                v = l.split()
                if len(v) >= 5 and peta.get(int(v[0])) is not None:
                    baris.append(' '.join([str(peta[int(v[0])])] + v[1:]))
        open(label_of(f'{OWN_DIR}/images/{os.path.basename(p)}'), 'w').write('\n'.join(baris))
SPLIT['own'] = f'{OWN_DIR}/images'
print('Foto test set sendiri:', len(list_images(SPLIT['own'])))

### 1.2 Statistik dataset

In [ ]:
def baca_label(img_path, w, h):
    # -> list (kelas, polygon Nx2 piksel). Baris bbox-only (4 angka) diubah jadi polygon persegi.
    out, lp = [], label_of(img_path)
    if not os.path.exists(lp): return out
    for l in open(lp):
        v = l.split()
        if len(v) < 5: continue
        c, xy = int(v[0]), np.array(v[1:], dtype=np.float32)
        if len(xy) == 4:
            cx, cy, bw, bh = xy
            xy = np.array([cx-bw/2, cy-bh/2, cx+bw/2, cy-bh/2, cx+bw/2, cy+bh/2, cx-bw/2, cy+bh/2], np.float32)
        if len(xy) < 6 or len(xy) % 2: continue
        out.append((c, xy.reshape(-1, 2) * [w, h]))
    return out

def hitung_instance(imgs):
    c = collections.Counter()
    for p in imgs:
        for l in open(label_of(p)) if os.path.exists(label_of(p)) else []:
            if l.strip(): c[NAMES[int(l.split()[0])]] += 1
    return c

stat = {s: hitung_instance(list_images(d)) for s, d in SPLIT.items()}
df_stat = pd.DataFrame(stat).reindex(NAMES).fillna(0).astype(int)
df_stat.loc['JUMLAH CITRA'] = [len(list_images(d)) for d in SPLIT.values()]
df_stat.to_csv(f'{BUKTI}/statistik_dataset.csv')
df_stat

### 1.3 Subset data latih 25% / 50% / 100% (variabel NRP)
Subset diambil acak dengan seed tetap dan **bersarang** (25% ⊂ 50% ⊂ 100%), sehingga perbedaan hasil hanya disebabkan oleh jumlah data. Validasi & test **tidak** dikurangi.

In [ ]:
train_all = list_images(SPLIT['train'])
acak = train_all[:]; random.Random(SEED).shuffle(acak)
DATA_YAML, TRAIN_LIST = {}, {}
for fr in FRACTIONS + MRCNN_FRACTIONS:
    tag = f'{int(fr*100)}'
    sub = sorted(acak[:max(1, round(len(acak) * fr))])
    TRAIN_LIST[fr] = sub
    txt = f'{WORK}/train_{tag}.txt'
    open(txt, 'w').write('\n'.join(os.path.abspath(p) for p in sub))
    DATA_YAML[fr] = f'{WORK}/data_{tag}.yaml'
    yaml.safe_dump(dict(train=txt, val=os.path.abspath(SPLIT['val']), test=os.path.abspath(SPLIT['test']),
                        nc=NC, names=NAMES), open(DATA_YAML[fr], 'w'), sort_keys=False)
YAML_OWN = f'{WORK}/data_own.yaml'
yaml.safe_dump(dict(train=os.path.abspath(SPLIT['train']), val=os.path.abspath(SPLIT['own']), test=os.path.abspath(SPLIT['own']),
                    nc=NC, names=NAMES), open(YAML_OWN, 'w'), sort_keys=False)
fr_urut = sorted(set(FRACTIONS + MRCNN_FRACTIONS))
df_sub = pd.DataFrame({f'{int(fr*100)}%': hitung_instance(TRAIN_LIST[fr]) for fr in fr_urut}).reindex(NAMES).fillna(0).astype(int)
df_sub.loc['JUMLAH CITRA'] = [len(TRAIN_LIST[fr]) for fr in fr_urut]
df_sub

### 1.4 Ground truth format COCO (dipakai untuk mengevaluasi semua model secara seragam)
YOLO dan Mask R-CNN punya fungsi evaluasi bawaan yang berbeda. Agar perbandingan adil, prediksi ketiga model dievaluasi dengan **COCOeval (`segm`)** terhadap ground truth yang sama.

In [ ]:
def buat_gt_coco(img_paths, out_json):
    images, anns, aid = [], [], 1
    for i, p in enumerate(img_paths, 1):
        h, w = cv2.imread(p).shape[:2]
        images.append(dict(id=i, file_name=os.path.abspath(p), width=w, height=h))
        for c, poly in baca_label(p, w, h):
            rle = maskU.merge(maskU.frPyObjects([poly.flatten().tolist()], h, w))
            area = float(maskU.area(rle))
            if area < 1: continue
            anns.append(dict(id=aid, image_id=i, category_id=c + 1, segmentation=[poly.flatten().tolist()],
                             area=area, bbox=maskU.toBbox(rle).tolist(), iscrowd=0))
            aid += 1
    json.dump(dict(images=images, annotations=anns, categories=[dict(id=k + 1, name=n) for k, n in enumerate(NAMES)]),
              open(out_json, 'w'))
    return out_json

GT = {s: buat_gt_coco(list_images(SPLIT[s]), f'{WORK}/gt_{s}.json') for s in ('val', 'test', 'own')}
IMGS = {s: list_images(SPLIT[s]) for s in ('val', 'test', 'own')}
print({s: len(v) for s, v in IMGS.items()})

### 1.5 Fungsi evaluasi bersama

In [ ]:
def ke_rle(m):
    r = maskU.encode(np.asfortranarray(m.astype(np.uint8)))
    r['counts'] = r['counts'].decode('ascii')
    return r

def evaluasi(gt_json, dets, img_ids=None, per_kelas=False):
    gt = COCO(gt_json) if isinstance(gt_json, str) else gt_json
    kosong = dict(mAP50_95=0.0, mAP50=0.0, mAP75=0.0, AP_kecil=0.0, AP_sedang=0.0, AP_besar=0.0, AR100=0.0)
    ids = set(img_ids) if img_ids is not None else None
    # salinan baru: COCO.loadRes MENGUBAH dict prediksi in-place (menambah bbox/area/id),
    # sehingga evaluasi kedua pada list yang sama akan error tanpa salinan ini.
    d = [dict(image_id=x['image_id'], category_id=x['category_id'], segmentation=x['segmentation'], score=x['score'])
         for x in dets if ids is None or x['image_id'] in ids]
    if not d:
        return (kosong, {}) if per_kelas else kosong
    with redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(d), 'segm')
        if img_ids is not None: E.params.imgIds = sorted(ids)
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    hasil = dict(mAP50_95=s[0], mAP50=s[1], mAP75=s[2], AP_kecil=s[3], AP_sedang=s[4], AP_besar=s[5], AR100=s[8])
    hasil = {k: max(float(v), 0.0) for k, v in hasil.items()}   # -1 (tidak ada objek) -> 0
    if not per_kelas: return hasil
    pk = {}
    for k, cid in enumerate(E.params.catIds):
        pr = E.eval['precision'][:, :, k, 0, 2]
        pk[gt.cats[cid]['name']] = float(pr[pr > -1].mean()) if (pr > -1).any() else np.nan
    return hasil, pk

# GT dimuat sekali, dipakai ulang
GT_COCO = {}
for s in GT:
    with redirect_stdout(io.StringIO()):
        GT_COCO[s] = COCO(GT[s])
ID_OF = {s: {im['file_name']: im['id'] for im in GT_COCO[s].dataset['images']} for s in GT}

## 2. Model keluarga one-stage: YOLO11n-seg & YOLOv8n-seg
Mask YOLO-seg = kombinasi linear dari **32 prototype mask** beresolusi `imgsz/4` (160×160 untuk input 640) dengan koefisien per deteksi, lalu dipotong dengan bounding box.

Eksperimen:
- `Y11_100`, `Y11_50`, `Y11_25` → YOLO11n-seg dengan 100/50/25% data latih (**variabel NRP**)
- `Y8_100` → YOLOv8n-seg, 100% data (model pembanding kedua)

Batas 2 jam dipasang lewat *callback* yang menghentikan training jika epoch berikutnya diperkirakan melewati batas.
> Catatan: parameter bawaan `time=2` di Ultralytics **tidak** cocok untuk ini — parameter itu justru menghitung ulang jumlah epoch agar training memakai penuh 2 jam.

In [ ]:
from ultralytics import YOLO

def pasang_batas_waktu(model, jam):
    def cek(trainer):
        if not hasattr(trainer, '_t_mulai_batas'):
            trainer._t_mulai_batas = trainer.train_time_start
        lewat = time.time() - trainer._t_mulai_batas
        per_epoch = lewat / max(1, trainer.epoch + 1)
        if lewat + per_epoch > jam * 3600:
            print(f'Batas {jam} jam GPU tercapai -> training dihentikan setelah epoch {trainer.epoch + 1}')
            trainer.stop = True
    model.add_callback('on_fit_epoch_end', cek)

YOLO_EXPS = [dict(nama=f'{k}_{int(fr*100)}', bobot=b, fr=fr)
             for k, b in [('Y11', 'yolo11n-seg.pt'), ('Y8', 'yolov8n-seg.pt')] for fr in FRACTIONS]
BOBOT = {}   # nama eksperimen -> path best weights

for e in YOLO_EXPS:
    best = f"{RUNS}/{e['nama']}/weights/best.pt"
    if os.path.exists(f"{RUNS}/{e['nama']}/SELESAI"):
        print(f"{e['nama']} sudah pernah dilatih -> dilewati ({best})"); BOBOT[e['nama']] = best; continue
    m = YOLO(e['bobot'])
    pasang_batas_waktu(m, BATAS_JAM_GPU)
    with catat_waktu_gpu(e['bobot'], e['nama'], f"data latih {int(e['fr']*100)}% ({len(TRAIN_LIST[e['fr']])} citra)"):
        m.train(data=DATA_YAML[e['fr']], epochs=EPOCHS_YOLO, imgsz=IMGSZ, batch=BATCH_YOLO, patience=15,
                seed=SEED, deterministic=True, project=RUNS, name=e['nama'], exist_ok=True,
                device=0 if DEVICE == 'cuda' else 'cpu', workers=2 if os.name != 'nt' else 0, plots=True, verbose=False)
    BOBOT[e['nama']] = str(m.trainer.best)
    open(f"{RUNS}/{e['nama']}/SELESAI", 'w').write(datetime.datetime.now().isoformat())
print(BOBOT)

## 3. Model keluarga two-stage: Mask R-CNN (ResNet-50-FPN v2, torchvision)
Tahap 1: RPN mengusulkan region → tahap 2: RoIAlign 14×14 → head klasifikasi+box dan **head mask yang memprediksi grid 28×28 per RoI per kelas**, lalu di-*resize* ke ukuran box.

Pengaturan disamakan dengan YOLO sebisanya: sisi terpanjang citra 640 px, bobot pretrained COCO, augmentasi flip horizontal. Log per-epoch ditulis ke `results.csv` (format serupa Ultralytics).

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

class SegDataset(torch.utils.data.Dataset):
    def __init__(self, imgs, train=False):
        self.imgs, self.train = imgs, train
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        p = self.imgs[i]
        im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = im.shape[:2]
        masks, labels = [], []
        for c, poly in baca_label(p, w, h):
            m = np.zeros((h, w), np.uint8)
            cv2.fillPoly(m, [np.round(poly).astype(np.int32)], 1)
            if m.sum() >= 4:
                masks.append(m); labels.append(c + 1)
        if self.train and random.random() < 0.5:
            im = im[:, ::-1]; masks = [m[:, ::-1] for m in masks]
        boxes = []
        for m in masks:
            ys, xs = np.where(m)
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        tgt = dict(boxes=torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                   labels=torch.tensor(labels, dtype=torch.int64),
                   masks=torch.from_numpy(np.stack(masks)) if masks else torch.zeros((0, h, w), dtype=torch.uint8))
        return torch.from_numpy(np.ascontiguousarray(im)).permute(2, 0, 1).float() / 255, tgt

def collate(b):
    return tuple(zip(*b))

def buat_mrcnn(pretrained=True):
    pretrained = pretrained and os.environ.get('T3_NOPRE') != '1'
    m = maskrcnn_resnet50_fpn_v2(weights='DEFAULT' if pretrained else None, weights_backbone=None,
                                 min_size=IMGSZ, max_size=IMGSZ, box_detections_per_img=100)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NC + 1)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(m.roi_heads.mask_predictor.conv5_mask.in_channels, 256, NC + 1)
    return m

@torch.no_grad()
def prediksi_mrcnn(model, split, pre=None, imgs=None):
    model.eval(); dets, t = [], 0.0
    for p in (imgs or IMGS[split]):
        im = cv2.imread(p)
        if pre is not None: im = pre(im)
        x = torch.from_numpy(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255).to(DEVICE)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.time(); o = model([x])[0]
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t += time.time() - t0
        mk = (o['masks'][:, 0] > 0.5).cpu().numpy()
        for k in range(len(o['scores'])):
            if mk[k].sum() == 0: continue
            dets.append(dict(image_id=ID_OF[split][os.path.abspath(p)], category_id=int(o['labels'][k]),
                             segmentation=ke_rle(mk[k]), score=float(o['scores'][k])))
    return dets, 1000 * t / max(1, len(imgs or IMGS[split]))

def latih_mrcnn(nama, train_imgs, epochs, batch, jam_maks):
    sd = f'{RUNS}/{nama}'; os.makedirs(f'{sd}/weights', exist_ok=True)
    dl = torch.utils.data.DataLoader(SegDataset(train_imgs, train=True), batch_size=batch, shuffle=True,
                                     num_workers=2 if os.name != 'nt' else 0, collate_fn=collate)
    model = buat_mrcnn().to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=0.02 * batch / 16, momentum=0.9, weight_decay=1e-4)
    total, warm = epochs * len(dl), min(500, len(dl))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda it: min(1, (it + 1) / warm) * 0.5 * (1 + np.cos(np.pi * it / total)))
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
    t_mulai, best, rows = time.time(), -1, []
    for ep in range(epochs):
        model.train(); jum = collections.defaultdict(float); n = 0
        for ims, tgts in dl:
            ims = [i.to(DEVICE) for i in ims]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                ld = model(ims, tgts)
                loss = sum(ld.values())
            if not torch.isfinite(loss):
                print('loss tidak finite, batch dilewati'); opt.zero_grad(); continue
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            for k, v in ld.items(): jum[k] += v.item()
            n += 1
        dets, _ = prediksi_mrcnn(model, 'val')
        ev = evaluasi(GT_COCO['val'], dets)
        row = dict(epoch=ep + 1, waktu_menit=round((time.time() - t_mulai) / 60, 2), lr=opt.param_groups[0]['lr'],
                   **{f'train/{k}': v / max(n, 1) for k, v in jum.items()},
                   **{'metrics/mAP50(M)': ev['mAP50'], 'metrics/mAP50-95(M)': ev['mAP50_95']})
        rows.append(row); pd.DataFrame(rows).to_csv(f'{sd}/results.csv', index=False)
        torch.save(model.state_dict(), f'{sd}/weights/last.pt')
        if ev['mAP50_95'] > best:
            best = ev['mAP50_95']; torch.save(model.state_dict(), f'{sd}/weights/best.pt')
        print(f"[{nama}] epoch {ep+1}/{epochs}  loss {sum(jum.values())/max(n,1):.3f}  val mask mAP50-95 {ev['mAP50_95']:.3f}  mAP50 {ev['mAP50']:.3f}")
        lewat = time.time() - t_mulai
        if lewat + lewat / (ep + 1) > jam_maks * 3600:
            print(f'Batas {jam_maks} jam GPU tercapai -> berhenti setelah epoch {ep+1}'); break
    return f'{sd}/weights/best.pt'

for fr in MRCNN_FRACTIONS:
    nama = f'MRCNN_{int(fr*100)}'
    best = f'{RUNS}/{nama}/weights/best.pt'
    if os.path.exists(f'{RUNS}/{nama}/SELESAI'):
        print(nama, 'sudah dilatih -> dilewati'); BOBOT[nama] = best; continue
    with catat_waktu_gpu('maskrcnn_resnet50_fpn_v2', nama, f'data latih {int(fr*100)}% ({len(TRAIN_LIST[fr])} citra)'):
        BOBOT[nama] = latih_mrcnn(nama, TRAIN_LIST[fr], EPOCHS_MRCNN, BATCH_MRCNN, BATAS_JAM_GPU)
    open(f'{RUNS}/{nama}/SELESAI', 'w').write(datetime.datetime.now().isoformat())

In [ ]:
# Kurva training (dari results.csv asli tiap run)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for nama in BOBOT:
    f = f'{RUNS}/{nama}/results.csv'
    if not os.path.exists(f): continue
    r = pd.read_csv(f); r.columns = [c.strip() for c in r.columns]
    ax[0].plot(r['epoch'], r['metrics/mAP50-95(M)'], label=nama)
    ax[1].plot(r['epoch'], r['metrics/mAP50(M)'], label=nama)
ax[0].set_title('Validasi mask mAP@50:95'); ax[1].set_title('Validasi mask mAP@50')
for a in ax: a.set_xlabel('epoch'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/gambar/kurva_training.png', dpi=150); plt.show()

## 4. Evaluasi pada dua test set: test asli vs test buatan sendiri (domain shift)

In [ ]:
def unletterbox(m, shape):
    # m: tensor mask (n, H, W) di ukuran inferensi (letterbox) -> tensor (n, h, w) di ukuran citra asli.
    # Dikerjakan di GPU. Versi numpy sebelumnya (cv2.resize + transpose) menghasilkan view berlompatan
    # sehingga thresholding per mask sangat lambat pada foto 1600 px (evaluasi >1 jam, GPU menganggur).
    H, W = m.shape[-2:]; h, w = shape
    g = min(H / h, W / w); pw, ph = (W - w * g) / 2, (H - h * g) / 2
    m = m[..., int(round(ph - .1)):int(round(H - ph + .1)), int(round(pw - .1)):int(round(W - pw + .1))]
    return torch.nn.functional.interpolate(m[None], size=(h, w), mode='bilinear', align_corners=False)[0]

def prediksi_yolo(model, split, imgsz=IMGSZ, retina=False, pre=None, imgs=None):
    dets, t = [], 0.0
    for p in (imgs or IMGS[split]):
        im = cv2.imread(p)
        if pre is not None: im = pre(im)
        t0 = time.time()
        r = model.predict(im, imgsz=imgsz, conf=0.001, max_det=100, retina_masks=retina, verbose=False,
                          device=0 if DEVICE == 'cuda' else 'cpu')[0]
        t += time.time() - t0
        if r.masks is None: continue
        md_ = r.masks.data.float()
        if md_.shape[-2:] != im.shape[:2]:
            md_ = unletterbox(md_, im.shape[:2])
        md_ = (md_ > 0.5).cpu().numpy()   # mask biner, contiguous per objek
        for k in range(len(r.boxes)):
            m = md_[k]
            if not m.any(): continue
            dets.append(dict(image_id=ID_OF[split][os.path.abspath(p)], category_id=int(r.boxes.cls[k]) + 1,
                             segmentation=ke_rle(m), score=float(r.boxes.conf[k])))
    return dets, 1000 * t / max(1, len(imgs or IMGS[split]))

def muat_model(nama):
    if nama.startswith('MRCNN'):
        m = buat_mrcnn(pretrained=False); m.load_state_dict(torch.load(BOBOT[nama], map_location=DEVICE)); return m.to(DEVICE).eval()
    return YOLO(BOBOT[nama])

def prediksi(nama, model, split, **kw):
    return prediksi_mrcnn(model, split, **kw) if nama.startswith('MRCNN') else prediksi_yolo(model, split, **kw)

DETS, HASIL, PER_KELAS = {}, [], {}
for nama in BOBOT:
    model = muat_model(nama)
    n_param = sum(p.numel() for p in (model.parameters() if nama.startswith('MRCNN') else model.model.parameters())) / 1e6
    row = dict(model=nama, params_M=round(n_param, 2))
    for s, lbl in [('test', 'asli'), ('own', 'sendiri')]:
        DETS[(nama, s)], ms = prediksi(nama, model, s)
        ev, pk = evaluasi(GT_COCO[s], DETS[(nama, s)], per_kelas=True)
        PER_KELAS[(nama, s)] = pk
        row.update({f'{k}_{lbl}': v for k, v in ev.items() if k in ('mAP50_95', 'mAP50', 'mAP75')})
        row[f'ms_per_img_{lbl}'] = ms
    HASIL.append(row)

tabel = pd.DataFrame(HASIL).set_index('model')
tabel['selisih_mAP50_95'] = tabel['mAP50_95_asli'] - tabel['mAP50_95_sendiri']
tabel['turun_relatif_%'] = 100 * tabel['selisih_mAP50_95'] / tabel['mAP50_95_asli'].replace(0, np.nan)
wl = waktu_tercatat()
if len(wl):
    tabel['waktu_latih_menit'] = wl.groupby('eksperimen')['durasi_menit'].sum().reindex(tabel.index)
tabel.to_csv(f'{BUKTI}/tabel_hasil_utama.csv')
tabel.round(3)

In [ ]:
# Cek silang: metrik bawaan Ultralytics untuk model YOLO (seharusnya mendekati angka COCOeval di atas)
cek = []
for nama in [n for n in BOBOT if not n.startswith('MRCNN')]:
    for s, y in [('test', DATA_YAML[1.0]), ('own', YAML_OWN)]:
        r = YOLO(BOBOT[nama]).val(data=y, split='test', imgsz=IMGSZ, verbose=False, plots=False,
                                  project=f'{RUNS}/val', name=f'{nama}_{s}', exist_ok=True)
        cek.append(dict(model=nama, test=s, mask_mAP50_95=r.seg.map, mask_mAP50=r.seg.map50))
pd.DataFrame(cek).round(3)

In [ ]:
pembanding = [n for n in BOBOT if n.endswith('_100')]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, met in zip(ax, ['mAP50_95', 'mAP50']):
    x = np.arange(len(pembanding))
    a.bar(x - .2, tabel.loc[pembanding, f'{met}_asli'], .4, label='test asli')
    a.bar(x + .2, tabel.loc[pembanding, f'{met}_sendiri'], .4, label='test sendiri')
    a.set_xticks(x, pembanding); a.set_title(f'Mask {met}'); a.legend(); a.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/gambar/perbandingan_model.png', dpi=150); plt.show()

### 4.1 Domain shift per kelas
Kelas mana yang paling turun AP-nya saat dipindah ke foto ruanganmu? Kelas yang tidak ada di test set sendiri bernilai NaN.

In [ ]:
dk = pd.DataFrame({f'{n}|{s}': PER_KELAS[(n, s)] for n in pembanding for s in ('test', 'own')}).reindex(NAMES)
for n in pembanding:
    dk[f'{n}|turun'] = dk[f'{n}|test'] - dk[f'{n}|own']
dk['instance_test_sendiri'] = [stat['own'].get(n, 0) for n in NAMES]
dk.to_csv(f'{BUKTI}/ap_per_kelas.csv')
dk.round(3).sort_values(f'{pembanding[0]}|turun', ascending=False)

### 4.2 Performa per kondisi foto (normal vs sulit)
Membaca `kondisi_foto.csv` hasil `siapkan_foto.py` (sudah kamu koreksi manual). Unggah file itu ke folder `OUT` (atau folder kerja Colab). Nama file dicocokkan walau Roboflow menambahkan akhiran `_jpg.rf.xxxx`.

In [ ]:
import re
KONDISI = None
for f in [f'{OUT}/kondisi_foto.csv', 'kondisi_foto.csv']:
    if os.path.exists(f):
        KONDISI = pd.read_csv(f); break
stem = lambda p: re.sub(r'_(jpg|jpeg|png|heic)\.rf\..*$', '', os.path.splitext(os.path.basename(p))[0], flags=re.I)
if KONDISI is None:
    print('kondisi_foto.csv tidak ditemukan -> bagian ini dilewati.')
else:
    peta_k = {os.path.splitext(f)[0]: k for f, k in zip(KONDISI['file'], KONDISI['kondisi'])}
    KOND_IMG = collections.defaultdict(list)
    for p in IMGS['own']:
        KOND_IMG[peta_k.get(stem(p), 'tidak_tercatat')].append(ID_OF['own'][os.path.abspath(p)])
    KOND_IMG['SEMUA_SULIT'] = [i for k, v in KOND_IMG.items() if k not in ('normal', 'tidak_tercatat') for i in v]
    print({k: len(v) for k, v in KOND_IMG.items()})
    dfk = pd.DataFrame({n: {k: evaluasi(GT_COCO['own'], DETS[(n, 'own')], ids)['mAP50'] for k, ids in KOND_IMG.items()}
                        for n in pembanding})
    dfk.insert(0, 'jumlah_foto', {k: len(v) for k, v in KOND_IMG.items()})
    dfk.to_csv(f'{BUKTI}/mAP50_per_kondisi.csv')
    display(dfk.round(3))

## 5. Variabel NRP: pengaruh jumlah data latih (25% / 50% / 100%) pada ketiga model
Pertanyaan yang dijawab: (1) seberapa besar mAP turun saat data dikurangi, (2) apakah **domain shift** (selisih test asli vs sendiri) mengecil atau membesar seiring bertambahnya data.

In [ ]:
KELUARGA = [('Y11', 'YOLO11n-seg', 'C0'), ('Y8', 'YOLOv8n-seg', 'C1'), ('MRCNN', 'Mask R-CNN', 'C2')]
PERSEN = sorted({int(f * 100) for f in FRACTIONS})

def nama_run(prefix):
    return [f'{prefix}_{p}' for p in PERSEN if f'{prefix}_{p}' in tabel.index]

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for prefix, label, warna in KELUARGA:
    ns = nama_run(prefix)
    if len(ns) < 2: continue
    x = [int(n.split('_')[-1]) for n in ns]
    ax[0].plot(x, tabel.loc[ns, 'mAP50_95_asli'], 'o-', c=warna, label=label)
    ax[1].plot(x, tabel.loc[ns, 'mAP50_95_sendiri'], 's--', c=warna, label=label)
    ax[2].plot(x, tabel.loc[ns, 'selisih_mAP50_95'], 'd-', c=warna, label=label)
for a, judul in zip(ax, ['Test asli', 'Test buatan sendiri', 'Domain shift (asli - sendiri)']):
    a.set_xticks(PERSEN); a.set_xlabel('% data latih'); a.set_title(judul); a.grid(alpha=.3); a.legend()
ax[0].set_ylabel('mask mAP@50:95'); ax[2].axhline(0, c='gray', lw=.8)
plt.tight_layout(); plt.savefig(f'{OUT}/gambar/variabel_nrp_jumlah_data.png', dpi=150); plt.show()

nrp = tabel.loc[[n for k, _, _ in KELUARGA for n in nama_run(k)],
                ['mAP50_95_asli', 'mAP50_asli', 'mAP50_95_sendiri', 'mAP50_sendiri', 'selisih_mAP50_95', 'waktu_latih_menit']]
nrp.to_csv(f'{BUKTI}/tabel_variabel_nrp.csv')
nrp.round(3)

## 6. Analisis kegagalan berbasis bukti
### 6.1 Diagnosis otomatis per foto (test set sendiri)
Setiap objek ground truth diklasifikasikan:
- **benar**: ada prediksi kelas sama dengan mask IoU ≥ 0.5
- **salah_kelas**: ada prediksi kelas *lain* dengan IoU ≥ 0.5
- **terpecah**: ≥2 prediksi kelas sama yang masing-masing ≥80% berada di dalam objek, gabungannya menutupi ≥50% objek
- **mask_bocor**: prediksi kelas sama dengan IoU 0.2–0.5 dan luasnya >1.3× objek (mask melebar ke latar/objek lain)
- **mask_kurang**: prediksi kelas sama IoU 0.2–0.5 tapi tidak melebar (tepi kasar / hanya sebagian)
- **tidak_terdeteksi**: selain itu

Prediksi dengan skor < 0.3 diabaikan pada diagnosis ini (seperti pemakaian nyata).

In [ ]:
def diagnosis(nama, split='own', skor_min=0.3):
    gt = GT_COCO[split]; baris = []
    for img in gt.dataset['images']:
        iid = img['id']
        g = gt.loadAnns(gt.getAnnIds(imgIds=iid))
        d = [x for x in DETS[(nama, split)] if x['image_id'] == iid and x['score'] >= skor_min]
        grle = [gt.annToRLE(a) for a in g]; drle = [x['segmentation'] for x in d]
        dc = np.array([x['category_id'] for x in d]); gc = np.array([a['category_id'] for a in g])
        if g and d:
            iou = maskU.iou(drle, grle, [0] * len(g))              # (n_pred, n_gt)
            inside = maskU.iou(drle, grle, [1] * len(g))           # irisan / luas prediksi
            garea = np.array([maskU.area(r) for r in grle]); darea = np.array([maskU.area(r) for r in drle])
        cnt = collections.Counter(); dipakai = set()
        urut = sorted(range(len(g)), key=lambda j: -(iou[:, j].max() if g and d else 0))
        for j in urut:
            if not d: cnt['tidak_terdeteksi'] += 1; continue
            sama = np.where(dc == gc[j])[0]
            kand = [k for k in sama if k not in dipakai and iou[k, j] >= 0.5]
            if kand:
                dipakai.add(max(kand, key=lambda k: iou[k, j])); cnt['benar'] += 1; continue
            lain = np.where((dc != gc[j]) & (iou[:, j] >= 0.5))[0]
            if len(lain): cnt['salah_kelas'] += 1; dipakai.add(lain[0]); continue
            pecah = [k for k in sama if inside[k, j] >= 0.8]
            if len(pecah) >= 2:
                cov = maskU.area(maskU.merge([drle[k] for k in pecah])) / garea[j]
                if cov >= 0.5: cnt['terpecah'] += 1; dipakai.update(pecah); continue
            sedang = [k for k in sama if 0.2 <= iou[k, j] < 0.5]
            if sedang:
                k = max(sedang, key=lambda k: iou[k, j]); dipakai.add(k)
                cnt['mask_bocor' if darea[k] > 1.3 * garea[j] else 'mask_kurang'] += 1; continue
            cnt['tidak_terdeteksi'] += 1
        cnt['prediksi_salah(FP)'] = len(d) - len(dipakai)
        tp = cnt['benar']; fn = len(g) - tp
        baris.append(dict(file=os.path.basename(img['file_name']), image_id=iid, n_objek=len(g),
                          F1=2 * tp / max(1, 2 * tp + fn + cnt['prediksi_salah(FP)']), **cnt))
    return pd.DataFrame(baris).fillna(0).sort_values('F1')

MODEL_UTAMA = tabel.loc[pembanding, 'mAP50_95_sendiri'].idxmax()   # ganti manual jika perlu
DIAG = {n: diagnosis(n) for n in pembanding}
print('Model yang dianalisis:', MODEL_UTAMA)
kol = ['n_objek', 'benar', 'tidak_terdeteksi', 'salah_kelas', 'terpecah', 'mask_bocor', 'mask_kurang', 'prediksi_salah(FP)']
display(pd.DataFrame({n: DIAG[n].reindex(columns=kol).sum() for n in pembanding}).T.astype(int))
DIAG[MODEL_UTAMA].to_csv(f'{BUKTI}/diagnosis_per_foto_{MODEL_UTAMA}.csv', index=False)
DIAG[MODEL_UTAMA].head(10)

### 6.2 Visualisasi foto terburuk: GT vs prediksi ketiga model
Pilih **5 foto** dari sini untuk laporan. Untuk tiap foto, kaitkan penyebab dengan mekanisme model, misalnya:
- tepi mask bergerigi/membulat pada objek besar → grid **28×28** Mask R-CNN yang di-*upsample* ke ukuran box
- objek tipis/kecil hilang atau mask "meleleh" → **prototype 160×160** YOLO (1 piksel prototype = 4 piksel input 640, lebih kasar lagi relatif terhadap foto 1600 px)
- mask terpotong lurus → mask YOLO & Mask R-CNN dipotong oleh **bounding box**; box yang terlalu kecil memotong mask
- objek bertumpuk jadi satu/terpecah → NMS dan matching anchor/RoI
- salah kelas di cahaya redup → distribusi warna/noise ISO tinggi tidak ada di data latih (domain shift)

In [ ]:
WARNA = plt.cm.tab20(np.linspace(0, 1, 20))[:, :3]

def overlay(im, anns, judul, ax, skor_min=0.3):
    vis = im.copy().astype(np.float32)
    for a in anns:
        if a.get('score', 1) < skor_min: continue
        m = maskU.decode(a['segmentation']) if isinstance(a['segmentation'], dict) else GT_COCO['own'].annToMask(a)
        c = WARNA[(a['category_id'] - 1) % 20] * 255
        vis[m > 0] = vis[m > 0] * 0.45 + c * 0.55
        ys, xs = np.where(m)
        if len(xs):
            ax.text(xs.mean(), ys.mean(), NAMES[a['category_id'] - 1] + (f" {a['score']:.2f}" if 'score' in a else ''),
                    fontsize=6, color='white', ha='center', bbox=dict(fc='black', alpha=.5, pad=1, lw=0))
    ax.imshow(vis.astype(np.uint8)); ax.set_title(judul, fontsize=9); ax.axis('off')

def tampilkan(iid, simpan=True):
    info = GT_COCO['own'].imgs[iid]
    im = cv2.cvtColor(cv2.imread(info['file_name']), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1 + len(pembanding), figsize=(4.2 * (1 + len(pembanding)), 4.5))
    overlay(im, GT_COCO['own'].loadAnns(GT_COCO['own'].getAnnIds(imgIds=iid)), 'Ground truth', ax[0])
    for a_, n in zip(ax[1:], pembanding):
        r = DIAG[n].set_index('image_id').loc[iid]
        overlay(im, [d for d in DETS[(n, 'own')] if d['image_id'] == iid], f'{n}  F1={r.F1:.2f}', a_)
    plt.suptitle(os.path.basename(info['file_name']), fontsize=9); plt.tight_layout()
    if simpan: plt.savefig(f"{OUT}/gambar/gagal_{stem(info['file_name'])}.png", dpi=130)
    plt.show()

for iid in DIAG[MODEL_UTAMA]['image_id'].head(8):
    tampilkan(iid)

### 6.3 Eksperimen kecil untuk menguji penjelasan
Jangan hanya berspekulasi — pilih eksperimen yang sesuai dengan hipotesismu dan jalankan. Semua eksperimen di bawah **tanpa training ulang** (murah GPU).

**H1 – Resolusi mask membatasi ketepatan tepi.** Prototype YOLO berukuran `imgsz/4`. Jika hipotesis benar, menaikkan resolusi inferensi (640→960, prototype 160→240) dan/atau `retina_masks=True` (upsample sebelum threshold) akan menaikkan **mAP75** dan **IoU rata-rata objek yang terdeteksi**, terutama pada objek besar. Untuk Mask R-CNN (grid 28×28 tetap), kita cek apakah IoU objek terdeteksi **turun pada objek besar** (1 sel grid mewakili makin banyak piksel).

**H2 – Kegagalan di foto redup/backlight disebabkan distribusi cahaya.** Jika benar, koreksi cahaya sederhana (CLAHE + gamma) pada foto sulit akan menaikkan mAP di subset itu tanpa mengubah model.

In [ ]:
def iou_objek_terdeteksi(dets, split='own', skor_min=0.3):
    # IoU tiap objek GT dengan prediksi terbaik kelas sama (hanya yang IoU>=0.5) + luas objek
    gt = GT_COCO[split]; out = []
    per_img = collections.defaultdict(list)
    for d in dets:
        if d['score'] >= skor_min: per_img[d['image_id']].append(d)
    for iid in gt.getImgIds():
        g = gt.loadAnns(gt.getAnnIds(imgIds=iid)); d = per_img[iid]
        if not g or not d: continue
        iou = maskU.iou([x['segmentation'] for x in d], [gt.annToRLE(a) for a in g], [0] * len(g))
        for j, a in enumerate(g):
            s = [iou[k, j] for k in range(len(d)) if d[k]['category_id'] == a['category_id']]
            if s and max(s) >= 0.5: out.append(dict(iou=max(s), area_frac=a['area'] / (gt.imgs[iid]['width'] * gt.imgs[iid]['height'])))
    return pd.DataFrame(out)

# ---------- H1: resolusi mask ----------
h1 = []
yolo_utama = [n for n in pembanding if not n.startswith('MRCNN')][0]
my = YOLO(BOBOT[yolo_utama])
BESAR = int(round(IMGSZ * 1.5 / 32) * 32)   # 640 -> 960
for imgsz, retina in [(IMGSZ, False), (IMGSZ, True), (BESAR, False), (BESAR, True)]:
    d, _ = prediksi_yolo(my, 'own', imgsz=imgsz, retina=retina)
    ev = evaluasi(GT_COCO['own'], d); io_ = iou_objek_terdeteksi(d)
    h1.append(dict(varian=f'{yolo_utama} imgsz={imgsz} retina={retina}', prototype=f'{imgsz//4}x{imgsz//4}',
                   mAP50=ev['mAP50'], mAP75=ev['mAP75'], mAP50_95=ev['mAP50_95'],
                   IoU_rata2_terdeteksi=io_['iou'].mean() if len(io_) else np.nan))
display(pd.DataFrame(h1).round(3))

# IoU vs ukuran objek: YOLO vs Mask R-CNN
fig, ax = plt.subplots(figsize=(7, 4))
bins = [0, .01, .05, .15, 1]; lab = ['<1%', '1-5%', '5-15%', '>15%']
for n in pembanding:
    io_ = iou_objek_terdeteksi(DETS[(n, 'own')])
    if len(io_):
        g = io_.groupby(pd.cut(io_['area_frac'], bins, labels=lab), observed=False)['iou'].mean()
        ax.plot(lab, g.values, 'o-', label=n)
ax.set_xlabel('luas objek (% luas citra)'); ax.set_ylabel('IoU rata-rata objek terdeteksi'); ax.legend(); ax.grid(alpha=.3)
ax.set_title('H1: ketepatan mask vs ukuran objek'); plt.tight_layout(); plt.savefig(f'{OUT}/gambar/H1_iou_vs_ukuran.png', dpi=150); plt.show()

In [ ]:
# ---------- H2: koreksi cahaya pada foto sulit ----------
def koreksi_cahaya(bgr, gamma=0.6):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    out = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return np.clip(255 * (out / 255.0) ** gamma, 0, 255).astype(np.uint8)

if KONDISI is None:
    print('Butuh kondisi_foto.csv untuk memilih foto sulit.')
else:
    target = [k for k in ('redup', 'backlight') if KOND_IMG.get(k)]
    ids = [i for k in target for i in KOND_IMG[k]]
    imgs_sulit = [GT_COCO['own'].imgs[i]['file_name'] for i in ids]
    h2 = []
    for n in pembanding:
        model = muat_model(n)
        for nm, pre in [('asli', None), ('CLAHE+gamma', koreksi_cahaya)]:
            d, _ = prediksi(n, model, 'own', pre=pre, imgs=imgs_sulit)
            ev = evaluasi(GT_COCO['own'], d, ids)
            h2.append(dict(model=n, input=nm, n_foto=len(ids), mAP50=ev['mAP50'], mAP50_95=ev['mAP50_95']))
    display(pd.DataFrame(h2).pivot(index='model', columns='input', values=['mAP50', 'mAP50_95']).round(3))
    if imgs_sulit:
        im = cv2.imread(imgs_sulit[0])
        plt.figure(figsize=(9, 4))
        plt.subplot(1, 2, 1); plt.imshow(im[..., ::-1]); plt.title('asli'); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(koreksi_cahaya(im)[..., ::-1]); plt.title('CLAHE + gamma'); plt.axis('off'); plt.show()

## 7. Simpan bukti proses
`results.csv` asli tiap run, log waktu GPU, tabel hasil, dan gambar dikumpulkan ke `bukti/` lalu dikompres.

In [ ]:
for nama in BOBOT:
    f = f'{RUNS}/{nama}/results.csv'
    if os.path.exists(f): shutil.copy(f, f'{BUKTI}/results_{nama}.csv')
print(waktu_tercatat())
print('\nTotal waktu GPU (menit):', waktu_tercatat()['durasi_menit'].sum().round(1) if len(waktu_tercatat()) else 0)
arsip = shutil.make_archive(f'{OUT}/bukti_tugas3', 'zip', OUT, 'bukti')
arsip_g = shutil.make_archive(f'{OUT}/gambar_tugas3', 'zip', OUT, 'gambar')
print(arsip, arsip_g)

## 8. Demo (sesi tanya jawab)
Inferensi langsung pada foto baru. Ubah `FOTO_DEMO`, `MODEL_DEMO`, dan `CONF_DEMO` lalu jalankan ulang sel ini.
Di Colab, foto bisa diunggah lewat panel **Files** (ikon folder di kiri), lalu salin path-nya ke `FOTO_DEMO`.

In [ ]:
FOTO_DEMO  = 'foto_demo.jpg'   # path foto baru (unggah dulu ke Colab)
MODEL_DEMO = 'Y11_100'         # pilih dari: list(BOBOT)
CONF_DEMO  = 0.25              # threshold skor: naikkan -> lebih sedikit tapi lebih yakin, turunkan -> lebih banyak

def demo(foto, nama=MODEL_DEMO, conf=CONF_DEMO):
    im = cv2.imread(foto)
    assert im is not None, f'Foto tidak ditemukan: {foto}'
    t0 = time.time()
    if nama.startswith('MRCNN'):
        model = muat_model(nama)
        with torch.no_grad():
            o = model([torch.from_numpy(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255).to(DEVICE)])[0]
        keep = o['scores'] >= conf
        masks = (o['masks'][keep, 0] > 0.5).cpu().numpy()
        cls = (o['labels'][keep] - 1).cpu().numpy(); skor = o['scores'][keep].cpu().numpy()
    else:
        r = YOLO(BOBOT[nama]).predict(im, imgsz=IMGSZ, conf=conf, retina_masks=True, verbose=False,
                                      device=0 if DEVICE == 'cuda' else 'cpu')[0]
        masks = r.masks.data.cpu().numpy() > 0.5 if r.masks is not None else np.zeros((0,) + im.shape[:2], bool)
        cls = r.boxes.cls.cpu().numpy().astype(int); skor = r.boxes.conf.cpu().numpy()
    ms = 1000 * (time.time() - t0)
    anns = [dict(segmentation=ke_rle(m), category_id=int(c) + 1, score=float(s)) for m, c, s in zip(masks, cls, skor)]
    fig, ax = plt.subplots(figsize=(9, 7))
    overlay(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), anns, f'{nama}  conf>={conf}  {len(anns)} objek  {ms:.0f} ms', ax, skor_min=conf)
    plt.show()
    return pd.DataFrame(dict(kelas=[NAMES[c] for c in cls], skor=np.round(skor, 3))).sort_values('skor', ascending=False)

if os.path.exists(FOTO_DEMO):
    display(demo(FOTO_DEMO))
else:
    print('Unggah foto lalu isi FOTO_DEMO. Contoh pakai foto test sendiri:')
    display(demo(IMGS['own'][0]))

In [ ]:
# AP satu kelas tertentu (mis. diminta dosen: "tampilkan AP kelas chair")
def ap_kelas(kelas, split='own'):
    return pd.DataFrame({n: {'AP@50:95': PER_KELAS[(n, split)].get(kelas, np.nan)} for n in BOBOT if (n, split) in PER_KELAS}).T.round(3)

ap_kelas('chair')